In [ ]:
# %%
# =============================================================================
# STEP 0: IMPORTS AND GEE INITIALIZATION
# =============================================================================

import ee
import geemap
import csv
from datetime import datetime, timedelta

try:
    ee.Initialize()
    print("✓ GEE initialized successfully")
except Exception:
    try:
        geemap.ee_initialize()
        ee.Initialize()
        print("✓ GEE initialized via geemap")
    except Exception:
        ee.Authenticate()
        ee.Initialize()
        print("✓ GEE initialized after authentication")

# --- Paths to Part 1 GeoTIFF exports (update these to your local paths) ---
# These are OPTIONAL — the script works without them, but having them
# enables offline analysis in matplotlib/QGIS later.
# Download from Google Drive → 'haor_flood_analysis' folder
GEOTIFF_DIR = "/work/a06/wasif/haor_flood_analysis"  # ← UPDATE THIS
FIRST_WET_DATE_TIF = f"{GEOTIFF_DIR}/first_wet_date_2025.tif"
WET_FRACTION_TIF   = f"{GEOTIFF_DIR}/wet_fraction_2025.tif"
TOTAL_WET_COUNT_TIF = f"{GEOTIFF_DIR}/total_wet_count_2025.tif"

✓ GEE initialized successfully


In [3]:
# =============================================================================
# STEP 1: RECREATE PART 1 CORE OBJECTS IN GEE
# =============================================================================
# Why: Even though Part 2 runs independently, we need the same GEE objects
#      (roi, flood_masks, first_wet_date) to compute per-haor statistics.
#      This step recreates them using the exact same parameters as Part 1.
#      It does NOT re-download or re-export anything — it just tells GEE
#      "here's the same computation I did before, keep it ready."

# --- Same parameters as Part 1 ---
BBOX = [89.8, 23.5, 92.8, 25.5]
roi = ee.Geometry.Rectangle(BBOX)

BASELINE_START = "2025-01-01"
BASELINE_END   = "2025-03-31"
MONITOR_START  = "2025-04-01"
MONITOR_END    = "2025-09-30"
Z_THRESHOLD    = -2.5
SLOPE_MAX      = 5

# --- Recreate S-1 collection and flood masks ---
def get_s1_collection(roi, start, end, pol="VV"):
    return (ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterBounds(roi)
            .filterDate(start, end)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", pol))
            .select(pol))

# Baseline stats
baseline_col = get_s1_collection(roi, BASELINE_START, BASELINE_END)
baseline_col = baseline_col.map(lambda img: img.updateMask(img.gt(-30)))
baseline_mean = baseline_col.mean().rename("baseline_mean")
baseline_std = baseline_col.reduce(ee.Reducer.stdDev()).rename("baseline_std")
baseline_std = baseline_std.where(baseline_std.lte(0), 0.5)

# Slope mask
slope = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003"))
slope_mask = slope.lt(SLOPE_MAX)

# Flood masks
def compute_zscore_flood_mask(image):
    masked = image.updateMask(image.gt(-30))
    zscore = masked.subtract(baseline_mean).divide(baseline_std)
    flood = zscore.lt(Z_THRESHOLD).rename("flood_clean")
    flood = flood.updateMask(slope_mask)
    flood = flood.focal_mode(radius=60, units="meters").rename("flood_clean")
    return flood.copyProperties(image, ["system:time_start"])

monitor_col = get_s1_collection(roi, MONITOR_START, MONITOR_END)
flood_masks = monitor_col.map(compute_zscore_flood_mask)

# First-wet-date
def add_date_band(img):
    date_ms = img.date().millis()
    epoch_ms = ee.Date("2025-01-01").millis()
    days = ee.Number(date_ms).subtract(epoch_ms).divide(86400000)
    date_img = ee.Image.constant(days).float().rename("date_days")
    return date_img.updateMask(img.select("flood_clean"))

first_wet_date = flood_masks.map(add_date_band).min().rename("first_wet_date")

# Get monitoring dates
monitor_dates_ms = flood_masks.aggregate_array("system:time_start").getInfo()
monitor_dates = sorted(set([
    datetime.utcfromtimestamp(d/1000).strftime("%Y-%m-%d") for d in monitor_dates_ms
]))

print("✓ Part 1 objects recreated in GEE")
print(f"  ROI: {BBOX}")
print(f"  Flood masks: {flood_masks.size().getInfo()} images")
print(f"  Monitoring dates: {len(monitor_dates)}")

/tmp/ipykernel_1280897/3292347227.py:66: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  datetime.utcfromtimestamp(d/1000).strftime("%Y-%m-%d") for d in monitor_dates_ms


✓ Part 1 objects recreated in GEE
  ROI: [89.8, 23.5, 92.8, 25.5]
  Flood masks: 175 images
  Monitoring dates: 71


In [4]:
# =============================================================================
# STEP 2 (FIXED): CoCoAH — HAOR BOUNDARY DETECTION
# =============================================================================
# PROBLEM in v1: Used reduceToVectors on raw S-1 water mask → fragmented,
#   noisy polygons scattered everywhere. Missed many haors, included noise.
#
# FIX: Use a HYBRID approach combining two data sources:
#
#   A) JRC Global Surface Water (Pekel et al. 2016)
#      - Pre-computed water occurrence from 1984-2021
#      - "occurrence" band: % of time each pixel was water (0-100)
#      - Pixels with occurrence 10-90% are SEASONAL water → likely haors
#      - Pixels with occurrence >90% are PERMANENT water → rivers, beels
#      - This gives us a clean, noise-free baseline of where haors typically form
#
#   B) Sentinel-1 May 2025 composite
#      - Shows the CURRENT state of haors in the pre-monsoon season
#      - Confirms which JRC-identified haors actually have water in 2025
#
#   By combining both, we get much cleaner boundaries than either alone:
#   - JRC removes SAR speckle noise (uses 36 years of Landsat data)
#   - S-1 confirms current-year water presence (JRC stops at 2021)
#
# SHAPE FILTERING: We implement Ahmad et al.'s eccentricity filter to
#   remove rivers (long thin features) from the haor candidates.

print("\n--- CoCoAH Haor Boundary Detection (Improved) ---")

# --- Source A: JRC Global Surface Water ---
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")

# "occurrence" = percentage of time water was present (0-100)
# "seasonality" = number of months water is present per year
water_occurrence = jrc.select("occurrence").clip(roi)
water_seasonality = jrc.select("seasonality").clip(roi)

# Seasonal water bodies: present 10-90% of the time → haor candidates
# This excludes permanent rivers (>90%) and rare puddles (<10%)
seasonal_water = water_occurrence.gte(10).And(water_occurrence.lte(90))

# Also include areas with clear seasonal pattern (present 4-10 months/year)
# Haors are typically inundated 6-8 months, dry 4-6 months
seasonal_by_months = water_seasonality.gte(4).And(water_seasonality.lte(10))

# Combine: pixel is a haor candidate if EITHER criterion is met
haor_candidate_jrc = seasonal_water.Or(seasonal_by_months).rename("haor_candidate")

print(f"  JRC seasonal water pixels identified")

# --- Source B: Sentinel-1 May 2025 composite ---
cocoah_col = get_s1_collection(roi, "2025-05-01", "2025-05-31")
cocoah_col = cocoah_col.map(lambda img: img.updateMask(img.gt(-30)))
cocoah_composite = cocoah_col.median().clip(roi)
n_cocoah = cocoah_col.size().getInfo()
print(f"  S-1 May 2025 images: {n_cocoah}")

# Water classification from S-1 (same -13dB threshold as Ahmad et al.)
s1_water_may = cocoah_composite.lt(-13).rename("s1_water")

# --- Combine: JRC haor candidate AND currently has water in May 2025 ---
# This is more conservative (fewer false positives) than either alone
haor_combined = haor_candidate_jrc.And(s1_water_may).selfMask()

# --- Morphological cleaning ---
# Use PROPER opening (erosion then dilation) to:
#   1. Remove thin connecting channels between haors (breaks river connections)
#   2. Remove small noise patches
#   3. Smooth haor boundaries
# Radius of 200m is appropriate: smaller than haors, larger than channels
eroded = haor_combined.focal_min(
    radius=200, units="meters", kernelType="circle"
)
dilated = eroded.focal_max(
    radius=200, units="meters", kernelType="circle"
)

# Fill internal holes (closing operation: dilation then erosion)
# This fills small dry patches inside haors (e.g., islands, missed pixels)
closed = dilated.focal_max(
    radius=150, units="meters", kernelType="circle"
).focal_min(
    radius=150, units="meters", kernelType="circle"
)

# --- Convert to vectors ---
COCOAH_SCALE = 100  # 100m for vectorization

print("  Vectorizing haor polygons...")
water_for_vector = closed.rename("water").selfMask()

haor_vectors_raw = water_for_vector.reduceToVectors(
    reducer=ee.Reducer.countEvery(),
    geometry=roi,
    scale=COCOAH_SCALE,
    maxPixels=1e12,
    geometryType="polygon",
    eightConnected=True,
    bestEffort=True
)

n_raw = haor_vectors_raw.size().getInfo()
print(f"  Raw polygons: {n_raw}")

# --- Size and shape filtering ---
# At 100m resolution, 1 pixel = 0.01 km²
# Minimum haor: 1 km² = 100 pixels (exclude small puddles)
# Maximum haor: 500 km² = 50000 pixels
MIN_HAOR_PIXELS = 100   # 1 km²
MAX_HAOR_PIXELS = 50000  # 500 km²

haor_vectors_sized = haor_vectors_raw.filter(
    ee.Filter.And(
        ee.Filter.gte("count", MIN_HAOR_PIXELS),
        ee.Filter.lte("count", MAX_HAOR_PIXELS)
    )
)

# --- Shape filtering: remove rivers using perimeter/area ratio ---
# Rivers are long and thin → high perimeter relative to area
# Haors are compact → lower perimeter relative to area
# We compute a "compactness" score: 4π × area / perimeter²
# Perfect circle = 1.0, long thin river ≈ 0.01-0.1
# Haors typically > 0.15, rivers typically < 0.1

def add_compactness(feature):
    """Compute compactness score for shape filtering."""
    area = feature.geometry().area(100)  # area in m²
    perimeter = feature.geometry().perimeter(100)  # perimeter in m
    # Compactness = 4π × area / perimeter²
    # Protect against zero perimeter
    compactness = ee.Number(4).multiply(3.14159).multiply(area).divide(
        perimeter.multiply(perimeter).max(1)
    )
    return feature.set("compactness", compactness)

haor_vectors_with_shape = haor_vectors_sized.map(add_compactness)

# Filter: keep only compact features (compactness > 0.08)
# This removes rivers and narrow channels while keeping haors
# 0.08 is lenient — adjust upward (e.g., 0.12) if rivers still sneak through
COMPACTNESS_THRESHOLD = 0.08

haor_vectors = haor_vectors_with_shape.filter(
    ee.Filter.gte("compactness", COMPACTNESS_THRESHOLD)
)

n_haors = haor_vectors.size().getInfo()
print(f"✓ CoCoAH detected {n_haors} haors (after size + shape filtering)")

# Get size distribution
haor_areas = haor_vectors.aggregate_array("count").getInfo()
if haor_areas:
    pixel_area_km2 = (COCOAH_SCALE / 1000.0) ** 2
    haor_areas_km2 = [a * pixel_area_km2 for a in haor_areas]
    print(f"  Size range: {min(haor_areas_km2):.1f} - {max(haor_areas_km2):.1f} km²")
    print(f"  Median: {sorted(haor_areas_km2)[len(haor_areas_km2)//2]:.1f} km²")
    print(f"  Total area: {sum(haor_areas_km2):.1f} km²")
    small = len([a for a in haor_areas_km2 if a < 5])
    medium = len([a for a in haor_areas_km2 if 5 <= a < 50])
    large = len([a for a in haor_areas_km2 if a >= 50])
    print(f"  Small (<5 km²): {small}, Medium (5-50): {medium}, Large (>50): {large}")

# Print compactness stats for inspection
compactness_vals = haor_vectors.aggregate_array("compactness").getInfo()
if compactness_vals:
    print(f"  Compactness range: {min(compactness_vals):.3f} - {max(compactness_vals):.3f}")



--- CoCoAH Haor Boundary Detection (Improved) ---
  JRC seasonal water pixels identified
  S-1 May 2025 images: 29
  Vectorizing haor polygons...
  Raw polygons: 1362
✓ CoCoAH detected 1345 haors (after size + shape filtering)
  Size range: 1.2 - 121.4 km²
  Median: 1.5 km²
  Total area: 4303.6 km²
  Small (<5 km²): 1187, Medium (5-50): 155, Large (>50): 3
  Compactness range: 0.085 - 0.644


In [5]:
# =============================================================================
# STEP 3: PER-HAOR FLOOD AREA TIME SERIES
# =============================================================================
# Why: The total area time series from Part 1 shows the overall flood hydrograph.
#      But to understand CONNECTIVITY between haors, you need to see WHEN each
#      individual haor floods. If Haor A peaks on May 20 and Haor B peaks on
#      June 5, that 16-day lag is evidence of propagation (or isolation).
#
# How: For each haor polygon from CoCoAH, compute the flooded area on each
#      monitoring date by masking the flood mask to that haor's boundary
#      and summing the wet pixels.
#
# This is computationally expensive for 700+ haors × 30+ dates.
# We do it for a SUBSET of the largest/most interesting haors first.

print("\n--- Per-Haor Flood Area Time Series ---")

# Select the top N haors by size for detailed analysis
# (Computing for ALL 700+ haors would take too long interactively)
TOP_N_HAORS = 20

# Sort haors by size (largest first) and take top N
haor_list = haor_vectors.toList(haor_vectors.size())

# We'll compute area time series for each selected haor
# First, let's get the geometries and IDs
print(f"  Computing area time series for top {TOP_N_HAORS} haors by size...")

# Create 6-day composite dates (same as Part 1 v2)
COMPOSITE_WINDOW_DAYS = 6
def generate_composite_dates(start_date, end_date, interval_days):
    dates = []
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    while current < end:
        dates.append(current.strftime("%Y-%m-%d"))
        current += timedelta(days=interval_days)
    return dates

composite_dates = generate_composite_dates(MONITOR_START, MONITOR_END, COMPOSITE_WINDOW_DAYS)

# For each of the top N haors, compute flooded area per composite date
per_haor_results = []

# Sort by area (count) descending
sorted_indices = sorted(range(len(haor_areas)), key=lambda i: haor_areas[i], reverse=True)

for rank, idx in enumerate(sorted_indices[:TOP_N_HAORS]):
    haor_feature = ee.Feature(haor_list.get(idx))
    haor_geom = haor_feature.geometry()
    haor_area_km2 = haor_areas[idx] * 0.01  # approximate total area in km²

    print(f"  Haor {rank+1}/{TOP_N_HAORS} (area ~{haor_area_km2:.1f} km²)...", end="", flush=True)

    haor_timeseries = []

    for date_str in composite_dates:
        end_dt = datetime.strptime(date_str, "%Y-%m-%d") + timedelta(days=COMPOSITE_WINDOW_DAYS)
        end_str = end_dt.strftime("%Y-%m-%d")

        # Get flood masks in this window
        window_masks = flood_masks.filterDate(date_str, end_str)
        n_imgs = window_masks.size().getInfo()

        if n_imgs > 0:
            # Union of flood detections in window
            composite = window_masks.select("flood_clean").max()
            # Compute flooded area within this haor
            flood_area = composite.multiply(ee.Image.pixelArea()).reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=haor_geom,
                scale=100,
                maxPixels=1e10
            ).get("flood_clean")

            area_val = ee.Number(flood_area).divide(1e6).getInfo()  # m² → km²
            haor_timeseries.append({
                "date": date_str,
                "flood_area_km2": round(area_val, 2) if area_val else 0
            })
        else:
            haor_timeseries.append({"date": date_str, "flood_area_km2": 0})

    per_haor_results.append({
        "haor_rank": rank + 1,
        "total_area_km2": haor_area_km2,
        "timeseries": haor_timeseries
    })
    print(" done")

print(f"\n✓ Per-haor area time series computed for {len(per_haor_results)} haors")

# Save to CSV
csv_file = "per_haor_area_timeseries_2025.csv"
with open(csv_file, "w", newline="") as f:
    writer = csv.writer(f)
    header = ["haor_rank", "total_area_km2"] + composite_dates
    writer.writerow(header)
    for h in per_haor_results:
        row = [h["haor_rank"], h["total_area_km2"]]
        ts_dict = {t["date"]: t["flood_area_km2"] for t in h["timeseries"]}
        row.extend([ts_dict.get(d, 0) for d in composite_dates])
        writer.writerow(row)
print(f"✓ Saved to {csv_file}")


--- Per-Haor Flood Area Time Series ---
  Computing area time series for top 20 haors by size...
  Haor 1/20 (area ~121.4 km²)... done
  Haor 2/20 (area ~72.1 km²)... done
  Haor 3/20 (area ~61.1 km²)... done
  Haor 4/20 (area ~49.3 km²)... done
  Haor 5/20 (area ~48.8 km²)... done
  Haor 6/20 (area ~43.1 km²)... done
  Haor 7/20 (area ~40.0 km²)... done
  Haor 8/20 (area ~36.6 km²)... done
  Haor 9/20 (area ~35.8 km²)... done
  Haor 10/20 (area ~34.9 km²)... done
  Haor 11/20 (area ~32.9 km²)... done
  Haor 12/20 (area ~31.3 km²)... done
  Haor 13/20 (area ~31.2 km²)... done
  Haor 14/20 (area ~30.8 km²)... done
  Haor 15/20 (area ~30.1 km²)... done
  Haor 16/20 (area ~27.3 km²)... done
  Haor 17/20 (area ~25.6 km²)... done
  Haor 18/20 (area ~25.1 km²)... done
  Haor 19/20 (area ~24.6 km²)... done
  Haor 20/20 (area ~24.0 km²)... done

✓ Per-haor area time series computed for 20 haors
✓ Saved to per_haor_area_timeseries_2025.csv


In [6]:
# =============================================================================
# STEP 4 (FIXED): MERIT HYDRO HAND + IMPROVED BARRIER DETECTION
# =============================================================================
# PROBLEM in v1: Simple HAND threshold of 2-8m captured almost everything
#   in this flat region → magenta dots everywhere, no useful information.
#
# FIX: Use a RELATIVE approach instead of absolute thresholds.
#   Instead of "HAND > 2m = barrier", we identify barriers as:
#   1. Pixels that are LOCALLY ELEVATED compared to their neighborhood
#   2. AND form LINEAR features (not random elevated patches)
#   3. AND are located BETWEEN haor polygons (not inside haors or on uplands)
#
# We combine MERIT Hydro HAND with FABDEM local relief for this.

print("\n--- MERIT Hydro HAND + Barrier Detection (Improved) ---")

merit_hydro = ee.Image("MERIT/Hydro/v1_0_1")
hand = merit_hydro.select("hnd").clip(roi)
flow_dir = merit_hydro.select("dir").clip(roi)
elev = merit_hydro.select("elv").clip(roi)
upa = merit_hydro.select("upa").clip(roi)

# HAND classification with TIGHTER thresholds for haor region
# In this extremely flat terrain:
#   HAND < 1m  → definitely in drainage/haor (very low-lying)
#   HAND 1-3m  → transitional (could be haor margin or low embankment)
#   HAND 3-6m  → likely elevated feature (embankment, road, settlement)
#   HAND > 6m  → definitely upland or hill
hand_classes = (ee.Image(0)
    .where(hand.lt(1), 1)          # Very low-lying (haor interior)
    .where(hand.gte(1).And(hand.lt(3)), 2)   # Transitional
    .where(hand.gte(3).And(hand.lt(6)), 3)   # Likely barrier
    .where(hand.gte(6), 4)         # Upland
    .rename("hand_class")
    .selfMask())

# River network from upstream area
river_network = upa.gt(100).rename("river_network")

# --- HAND-based barrier detection (more refined) ---
# Barriers should be:
# 1. Moderately elevated (HAND 1.5-6m — not too high, not too low)
# 2. Adjacent to low-lying areas on BOTH sides
# 3. Not part of the river network

# Step 1: Identify moderately elevated pixels
moderate_elevation = hand.gte(1.5).And(hand.lt(6))

# Step 2: Check if they're adjacent to low-lying areas
# A barrier should have low HAND values nearby (within ~500m)
nearby_min_hand = hand.focal_min(radius=500, units="meters")
near_low_areas = nearby_min_hand.lt(1.5)  # There's a low area within 500m

# Step 3: Exclude river network pixels
not_river = upa.lt(50)  # Not a major channel

# Combine: barrier candidate = elevated AND near low areas AND not river
hand_barriers = moderate_elevation.And(near_low_areas).And(not_river).rename("hand_barrier")

print("✓ MERIT Hydro layers loaded with refined barrier detection")
hand_stats = hand.reduceRegion(
    reducer=ee.Reducer.percentile([10, 25, 50, 75, 90]),
    geometry=roi, scale=90, maxPixels=1e10
).getInfo()
print(f"  HAND percentiles: {hand_stats}")


--- MERIT Hydro HAND + Barrier Detection (Improved) ---
✓ MERIT Hydro layers loaded with refined barrier detection
  HAND percentiles: {'hnd_p10': 1.5026750101328628, 'hnd_p25': 1.5026750101328628, 'hnd_p50': 1.5026750101328628, 'hnd_p75': 19.72117179037871, 'hnd_p90': 83.86540171630536}


In [7]:
# %%
# =============================================================================
# STEP 5 (FIXED): FABDEM 30m — BARRIER LIKELIHOOD LAYER
# =============================================================================

print("\n--- FABDEM 30m — Barrier Likelihood Layer ---")

fabdem = ee.ImageCollection("projects/sat-io/open-datasets/FABDEM")
fabdem_mosaic = fabdem.filterBounds(roi).mosaic().clip(roi).rename("fabdem_elev")

# --- Local relief at fine scale (200m) ---
focal_mean_200 = fabdem_mosaic.focal_mean(radius=200, units="meters")
local_relief_200 = fabdem_mosaic.subtract(focal_mean_200).rename("local_relief_200m")

# --- Local relief at medium scale (500m) ---
focal_mean_500 = fabdem_mosaic.focal_mean(radius=500, units="meters")
local_relief_500 = fabdem_mosaic.subtract(focal_mean_500).rename("local_relief_500m")

# --- Barrier detection from FABDEM ---
fabdem_barrier_fine = local_relief_200.gt(0.5)
fabdem_barrier_medium = local_relief_500.gt(0.8)
fabdem_barrier = fabdem_barrier_fine.Or(fabdem_barrier_medium).rename("fabdem_barrier")

# --- Combined barrier likelihood ---
barrier_both = hand_barriers.And(fabdem_barrier).rename("barrier_high_conf")
barrier_either = hand_barriers.Or(fabdem_barrier).rename("barrier_any_conf")

# --- Linear feature enhancement ---
# FIX: Use reduceNeighborhood instead of focal_sum (which doesn't exist in Python API)
kernel_small = ee.Kernel.circle(radius=100, units="meters")
kernel_large = ee.Kernel.circle(radius=300, units="meters")

barrier_count_small = barrier_either.reduceNeighborhood(
    reducer=ee.Reducer.sum(),
    kernel=kernel_small
)
barrier_count_large = barrier_either.reduceNeighborhood(
    reducer=ee.Reducer.sum(),
    kernel=kernel_large
)

# Ratio > 0.15 means barrier pixels are concentrated (linear-ish)
linearity_ratio = barrier_count_small.divide(barrier_count_large.max(1))
linear_barriers = barrier_either.And(linearity_ratio.gt(0.15)).rename("linear_barriers")

print("✓ FABDEM barrier likelihood computed")
print("  Layers: local_relief_200m, local_relief_500m, fabdem_barrier,")
print("          barrier_high_conf, barrier_any_conf, linear_barriers")

barrier_stats = barrier_both.selfMask().reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=roi, scale=100, maxPixels=1e10
).getInfo()
print(f"  High-confidence barrier pixels: {barrier_stats}")


--- FABDEM 30m — Barrier Likelihood Layer ---
✓ FABDEM barrier likelihood computed
  Layers: local_relief_200m, local_relief_500m, fabdem_barrier,
          barrier_high_conf, barrier_any_conf, linear_barriers
  High-confidence barrier pixels: {'barrier_high_conf': 354004}


In [8]:
# =============================================================================
# STEP 6: OSM ROAD NETWORK AS BARRIER PROXY
# =============================================================================
# Why: In the haor region, major roads and raised embankments act as physical
#      barriers that block or delay flood water. Roads in haor areas are
#      typically built on raised earth platforms (1-4m above the surrounding
#      paddy fields) and function as de facto levees.
#
# GEE doesn't have a dedicated "embankment" layer from OSM, but we can
# use a global road dataset. We use the GRIP (Global Roads Inventory Project)
# dataset, which is available on GEE and includes road classifications.
#
# For haor areas, even minor roads can act as significant barriers because
# the terrain is so flat — a 1-meter raised road bed is enough to block
# shallow flood water.

print("\n--- Road Network (Barrier Proxy) ---")

# Option 1: Use GRIP global roads dataset (available on GEE)
# This has good coverage of Bangladesh including rural roads
try:
    grip_roads = ee.FeatureCollection("projects/sat-io/open-datasets/GRIP4/GRIP4-region5")
    roads_in_roi = grip_roads.filterBounds(roi)
    n_roads = roads_in_roi.size().getInfo()
    print(f"✓ GRIP roads loaded: {n_roads} road segments in AOI")
    has_roads = True
except Exception as e:
    print(f"  GRIP roads not available: {e}")
    print("  Trying alternative road source...")

    # Option 2: Use a simplified approach — rasterize roads from
    # the FABDEM itself by detecting linear elevated features
    # (Roads appear as narrow ridges in the DEM)
    has_roads = False

if not has_roads:
    # Fallback: detect linear elevated features from FABDEM
    # Roads in flat haor terrain create detectable ridges
    print("  Using FABDEM-derived elevated linear features as barrier proxy")

    # Simple edge detection: where FABDEM elevation is locally higher
    # than its neighbors, there might be a road/embankment
    focal_mean = fabdem_mosaic.focal_mean(radius=150, units="meters")
    local_relief = fabdem_mosaic.subtract(focal_mean).rename("local_relief")

    # Pixels where local relief > 0.5m are potentially raised features
    raised_features = local_relief.gt(0.5).rename("raised_features")
    print("✓ Elevated linear features detected from FABDEM")



--- Road Network (Barrier Proxy) ---
  GRIP roads not available: Collection.loadTable: Collection asset 'projects/sat-io/open-datasets/GRIP4/GRIP4-region5' not found.
  Trying alternative road source...
  Using FABDEM-derived elevated linear features as barrier proxy
✓ Elevated linear features detected from FABDEM


In [9]:
# =============================================================================
# STEP 7: ELEVATION TRANSECTS ACROSS BARRIERS
# =============================================================================
# Why: To validate the levee-overtopping mechanism, you need to show that
#      there IS an elevation barrier between two haors. An elevation transect
#      is a cross-section showing ground elevation along a straight line
#      from Haor A across the barrier into Haor B.
#
# The transect should show: low (haor A) → high (barrier) → low (haor B)
#
# We define 2-3 sample transects across known barrier locations.
# You can adjust these coordinates after inspecting the first-wet-date map
# to target areas where you see abrupt changes in flood timing.

print("\n--- Elevation Transects ---")

# Define sample transect lines
# Each transect is defined by a start point and end point [lon, lat]
# These are placed across areas where roads/embankments separate haors
# ADJUST THESE after inspecting your first-wet-date map!
transects = [
    {
        "name": "Transect 1: Sunamganj N-S",
        "start": [91.0, 25.1],  # North of Sunamganj
        "end": [91.0, 24.7],    # South toward haor interior
        "description": "Crosses multiple haors and roads N-S through Sunamganj"
    },
    {
        "name": "Transect 2: Tanguar-Dekhar corridor",
        "start": [91.1, 25.15],  # Near Tanguar haor
        "end": [91.5, 25.0],     # Toward Dekhar haor
        "description": "Connects two well-known haors with embankments between"
    },
    {
        "name": "Transect 3: E-W across central haor belt",
        "start": [90.5, 24.8],   # Western haor region
        "end": [91.8, 24.8],     # Eastern haor region
        "description": "Long E-W transect crossing many haors and barriers"
    },
]

# Extract elevation profiles from FABDEM along each transect
transect_results = []

for t in transects:
    line = ee.Geometry.LineString([t["start"], t["end"]])

    # Sample FABDEM elevation along the transect at 30m intervals
    # Also sample HAND and first-wet-date for comparison
    profile_points = ee.Image.cat([
        fabdem_mosaic.rename("elevation"),
        hand.rename("hand"),
        first_wet_date.rename("first_wet_date")
    ]).sample(
        region=line,
        scale=30,
        numPixels=500,  # Sample up to 500 points along the line
        geometries=True  # Keep point locations
    )

    # Extract results
    profile_data = profile_points.getInfo()
    n_points = len(profile_data["features"])
    print(f"  {t['name']}: {n_points} points sampled")

    transect_results.append({
        "name": t["name"],
        "description": t["description"],
        "start": t["start"],
        "end": t["end"],
        "data": profile_data
    })

# Save transect data to CSV for plotting in matplotlib
for i, t in enumerate(transect_results):
    csv_name = f"transect_{i+1}_profile.csv"
    with open(csv_name, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["lon", "lat", "elevation_m", "hand_m", "first_wet_day"])
        for feat in t["data"]["features"]:
            props = feat["properties"]
            coords = feat["geometry"]["coordinates"]
            writer.writerow([
                round(coords[0], 6),
                round(coords[1], 6),
                round(props.get("elevation", -999), 2),
                round(props.get("hand", -999), 2),
                round(props.get("first_wet_date", -999), 1)
            ])
    print(f"  Saved to {csv_name}")

print(f"\n✓ {len(transect_results)} transect profiles extracted and saved")


--- Elevation Transects ---
  Transect 1: Sunamganj N-S: 969 points sampled
  Transect 2: Tanguar-Dekhar corridor: 1070 points sampled
  Transect 3: E-W across central haor belt: 2790 points sampled
  Saved to transect_1_profile.csv
  Saved to transect_2_profile.csv
  Saved to transect_3_profile.csv

✓ 3 transect profiles extracted and saved


In [10]:
# =============================================================================
# STEP 8 (UPDATED): COMBINED ANALYSIS MAP
# =============================================================================
# Updated to use the new improved layers from Steps 2, 4, 5.

print("\n--- Creating Combined Analysis Map (Improved) ---")
m = geemap.Map()
m.centerObject(roi, 8)

# 1. First-wet-date (from Part 1)
first_wet_vis = {
    "min": 90, "max": 272,
    "palette": [
        "023858", "045a8d", "0570b0", "3690c0",
        "74a9cf", "66c2a4", "41ae76", "238b45",
        "006d2c", "31a354", "78c679", "addd8e",
        "d9f0a3", "f7fcb1", "ffeda0", "fed976",
        "feb24c", "fd8d3c", "fc4e2a", "e31a1c",
        "d7301f", "bd0026", "a50026", "800026",
    ]
}
m.addLayer(first_wet_date.clip(roi), first_wet_vis,
           "1. First Wet Date", True)

# 2. MERIT Hydro HAND
hand_vis = {
    "min": 0, "max": 10,
    "palette": ["0D0887", "5B02A3", "9A179B", "CB4678",
                "EB7852", "FBB32F", "F0F921"]
}
m.addLayer(hand.clip(roi), hand_vis, "2. HAND (m)", False)

# 3. HAND classification (refined)
hand_class_vis = {
    "min": 1, "max": 4,
    "palette": ["2166AC", "92C5DE", "F4A582", "B2182B"]
}
m.addLayer(hand_classes.clip(roi), hand_class_vis,
           "3. HAND classes (blue=low → red=high)", False)

# 4. FABDEM local relief (fine scale) — NEW
relief_vis = {
    "min": -1, "max": 2,
    "palette": ["0571B0", "92C5DE", "F7F7F7", "F4A582", "CA0020"]
}
m.addLayer(local_relief_200.clip(roi), relief_vis,
           "4. FABDEM local relief 200m (m)", False)

# 5. High-confidence barriers (HAND + FABDEM agree) — NEW
m.addLayer(barrier_both.selfMask().clip(roi),
           {"palette": ["FF00FF"]},
           "5. Barriers: HIGH confidence (HAND+FABDEM)", False)

# 6. Linear barriers — NEW
m.addLayer(linear_barriers.selfMask().clip(roi),
           {"palette": ["FFFF00"]},
           "6. Barriers: LINEAR features", False)

# 7. River network
m.addLayer(river_network.selfMask().clip(roi),
           {"palette": ["00BFFF"]},
           "7. River network (upa>100km²)", False)

# 8. CoCoAH haor boundaries — IMPROVED
haor_outline = ee.Image().paint(haor_vectors, 1, 2)
m.addLayer(haor_outline, {"palette": ["00FF00"]},
           "8. CoCoAH haor boundaries", True)

# 9. JRC water occurrence (reference)
jrc_vis = {"min": 0, "max": 100, "palette": ["FFFFFF", "0000FF"]}
m.addLayer(water_occurrence, jrc_vis,
           "9. JRC water occurrence (%)", False)

# 10. FABDEM raw elevation (for reference only)
fabdem_vis = {
    "min": 0, "max": 30,
    "palette": ["000004", "1B0C41", "4A0C6B", "781C6D",
                "A52C60", "CF4446", "ED6925", "FB9B06",
                "F7D03C", "FCFFA4"]
}
m.addLayer(fabdem_mosaic.clip(roi), fabdem_vis,
           "10. FABDEM elevation (m, reference)", False)

# Transect lines
transects = [
    {"name": "Transect 1: Sunamganj N-S", "start": [91.0, 25.1], "end": [91.0, 24.7]},
    {"name": "Transect 2: Tanguar-Dekhar", "start": [91.1, 25.15], "end": [91.5, 25.0]},
    {"name": "Transect 3: E-W central", "start": [90.5, 24.8], "end": [91.8, 24.8]},
]
for t in transects:
    line = ee.Geometry.LineString([t["start"], t["end"]])
    m.addLayer(ee.Image().paint(ee.FeatureCollection([ee.Feature(line)]), 1, 3),
               {"palette": ["FFFFFF"]}, f"T: {t['name']}", False)

# Study area
m.addLayer(ee.Image().paint(roi, 1, 2), {"palette": ["FFFFFF"]}, "Study Area", True)

print("✓ Combined analysis map created — run 'm' to display")
print("""
Layer guide:
  1. First Wet Date — flood propagation pattern (blue=early, red=late)
  2. HAND — height above nearest drainage (meters)
  3. HAND classes — 4 levels from very low to upland
  4. FABDEM local relief — how much higher than 200m neighborhood
     (red = elevated features like embankments, blue = depressions)
  5. HIGH confidence barriers — where BOTH HAND and FABDEM agree
     (most reliable barrier locations — magenta)
  6. LINEAR barriers — elevated features that form lines/ridges
     (likely roads or embankments — yellow)
  7. River network — major channels
  8. CoCoAH haor boundaries — improved using JRC+S-1 (green)
  9. JRC water occurrence — historical water frequency (reference)
  10. FABDEM elevation — raw terrain height (reference only)

Key analysis:
  - Toggle layers 1 + 8 together: do haor boundaries align with
    first-wet-date patterns?
  - Toggle layers 1 + 5 or 6: do barriers align with sharp changes
    in flood timing?
  - Toggle layers 4 + 8: do local relief ridges correspond to
    haor boundaries?
""")



--- Creating Combined Analysis Map (Improved) ---
✓ Combined analysis map created — run 'm' to display

Layer guide:
  1. First Wet Date — flood propagation pattern (blue=early, red=late)
  2. HAND — height above nearest drainage (meters)
  3. HAND classes — 4 levels from very low to upland
  4. FABDEM local relief — how much higher than 200m neighborhood
     (red = elevated features like embankments, blue = depressions)
  5. HIGH confidence barriers — where BOTH HAND and FABDEM agree
     (most reliable barrier locations — magenta)
  6. LINEAR barriers — elevated features that form lines/ridges
     (likely roads or embankments — yellow)
  7. River network — major channels
  8. CoCoAH haor boundaries — improved using JRC+S-1 (green)
  9. JRC water occurrence — historical water frequency (reference)
  10. FABDEM elevation — raw terrain height (reference only)

Key analysis:
  - Toggle layers 1 + 8 together: do haor boundaries align with
    first-wet-date patterns?
  - Toggle layers 

In [11]:
m

Map(center=[24.504754997740452, 91.30000000000003], controls=(WidgetControl(options=['position', 'transparent_…